# Paper Method Reproduction — Saha et al. 2025



## Section 0 — Setup


In [1]:
# Standard library
import os, sys, math, warnings
from pathlib import Path

# Data and ML libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import GroupKFold, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import ElasticNet
from sklearn.metrics import r2_score, mean_squared_error
from scipy.stats import ttest_rel

# Reduce noise from warnings
warnings.filterwarnings("ignore")

# Set working directory to the bundle root (one level above notebooks/)
BUNDLE_ROOT = Path("..").resolve()
os.chdir(BUNDLE_ROOT)
#os.chdir("/Users/robertwang/Documents/New project/baseline_composite_biomakers")
print("Working directory:", Path.cwd())

# Runtime control: set True for fast demo, False for full paper protocol
DEMO_MODE = True  # full protocol
RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)


Working directory: /Users/robertwang/Documents/New_project/baseline_composite_biomakers


## Section 2 — Feature Definitions and Combinations

In [2]:
# Feature domains (Melbourne v1 columns)
background = [
    'age1',
    'age2',
    'age_at_onset',
    'dur1',
    'dur2',
    'sex',
    'GAA1',
    'GAA2',
]

structural = [
    'SCP_v1', 'MCP_v1', 'ICP_v1',
    'Midbrain_v1', 'Pons_v1', 'Medulla_v1',
    'AntCBLM_v1', 'SupPostCBLM_v1', 'InfPostCBLM_v1',
    'FlocCBLM_v1', 'VermisCBLM_v1',
]

diffusion = [
    'FASCP_v1', 'FAMCP_v1', 'FAICP_v1',
    'MDSCP_v1', 'MDMCP_v1', 'MDICP_v1',
    'ADSCP_v1', 'ADMCP_v1', 'ADICP_v1',
    'RDSCP_v1', 'RDMCP_v1', 'RDICP_v1',
]

qsm = []  # not available

# Combination definitions (paper Table 3); QSM combos skipped
combinations = [
    {"name": "background_only", "domains": [background], "skip": False},
    {"name": "structural_only", "domains": [structural], "skip": False},
    {"name": "diffusion_only", "domains": [diffusion], "skip": False},
    {"name": "qsm_only", "domains": [qsm], "skip": True},
    {"name": "all_neuroimaging", "domains": [structural, diffusion], "skip": False},
    {"name": "background_structural", "domains": [background, structural], "skip": False},
    {"name": "background_diffusion", "domains": [background, diffusion], "skip": False},
    {"name": "background_qsm", "domains": [background, qsm], "skip": True},
    {"name": "background_structural_diffusion", "domains": [background, structural, diffusion], "skip": False},
    {"name": "background_all_neuroimaging", "domains": [background, structural, diffusion, qsm], "skip": True},
]

print('Defined combinations:')
for c in combinations:
    print(c['name'], 'skip' if c['skip'] else 'run', 'features:', sum(len(d) for d in c['domains']))


Defined combinations:
background_only run features: 8
structural_only run features: 11
diffusion_only run features: 12
qsm_only skip features: 0
all_neuroimaging run features: 23
background_structural run features: 19
background_diffusion run features: 20
background_qsm skip features: 8
background_structural_diffusion run features: 31
background_all_neuroimaging skip features: 31


## Section 3 — QC Checks (Missingness / Outliers)

In [3]:
# Load Melbourne-only merged data for baseline paper run (v1 only)
#melb_path = Path('/Users/robertwang/Documents/New project/baseline_composite_biomakers/data/processed/melbourne_frda_merged.csv')
melb_path = Path('data/processed/melbourne_frda_merged.csv')
assert melb_path.exists(), 'Run data_processing_merge.ipynb first to generate melbourne_frda_merged.csv'

merged_long = pd.read_csv(melb_path)
print('Loaded melbourne_frda_merged.csv:', merged_long.shape)



Loaded melbourne_frda_merged.csv: (26, 69)


In [4]:
# Missingness summary

def missingness_summary(df, cols):
    # Only keep columns that exist, warn if any are missing
    present = [c for c in cols if c in df.columns]
    missing = [c for c in cols if c not in df.columns]
    if missing:
        print("WARNING: missing columns:", missing)
    return df[present].isna().sum().sort_values(ascending=False)

print("Missingness (top 10):")
print(missingness_summary(merged_long, background + structural + diffusion).head(10))


Missingness (top 10):
age1              0
InfPostCBLM_v1    0
RDMCP_v1          0
RDSCP_v1          0
ADICP_v1          0
ADMCP_v1          0
ADSCP_v1          0
MDICP_v1          0
MDMCP_v1          0
MDSCP_v1          0
dtype: int64


In [5]:
# Tukey's Fences outlier filter (k=3) — features only

def tukey_outliers_mask(df, cols, k=3.0):
    mask = pd.Series(False, index=df.index)
    for col in cols:
        if col not in df.columns:
            continue
        series = df[col].dropna()
        if series.empty:
            continue
        q1 = series.quantile(0.25)
        q3 = series.quantile(0.75)
        iqr = q3 - q1
        lo = q1 - k * iqr
        hi = q3 + k * iqr
        mask |= (df[col] < lo) | (df[col] > hi)
    return mask


## Section 4 — Model Training (ElasticNet)
From paper protocol: 10×10 GroupKFold, grid over alpha (0..1) and lambda (0..20).


In [6]:
# Hyperparameter grid
if DEMO_MODE:
    alphas = np.linspace(0, 1, 3)        # tiny grid
    lambdas = np.linspace(0, 2, 5)
    n_splits = 3
    n_repeats = 2
else:
    alphas = np.linspace(0, 1, 11)       # 0..1 step 0.1
    lambdas = np.linspace(0, 20, 201)    # 0..20 step 0.1
    n_splits = 10
    n_repeats = 10

print("Grid size:", len(alphas) * len(lambdas))


Grid size: 15


In [7]:
# Cross-validation with per-fold hyperparameter search

def run_cv_for_combination(df, target_col, combo):
    feature_cols = []
    for domain in combo["domains"]:
        feature_cols.extend(domain)

    # Drop rows missing any features or target
    sub = df.dropna(subset=feature_cols + [target_col]).copy()
    n_before = len(sub)

    # Remove outliers (features only)
    outlier_mask = tukey_outliers_mask(sub, feature_cols, k=3.0)
    sub = sub[~outlier_mask].copy()
    n_after = len(sub)
    # Grouping column (subject ID)
    group_col = 'subject' if 'subject' in sub.columns else ('melb_id' if 'melb_id' in sub.columns else ('ID' if 'ID' in sub.columns else None))
    if group_col is None:
        raise KeyError('No subject/group id column found for GroupKFold')

    groups = sub[group_col].values
    X = sub[feature_cols].values
    y = sub[target_col].values

    all_preds = []
    all_true = []
    best_alphas = []
    best_l1s = []

    for r in range(n_repeats):
        rng = np.random.RandomState(RANDOM_SEED + r)
        unique_groups = np.unique(groups)
        rng.shuffle(unique_groups)

        # Shuffle rows by group order (paper repeats CV with reshuffled groups)
        group_order = {g: i for i, g in enumerate(unique_groups)}
        order = np.argsort([group_order[g] for g in groups])
        Xr, yr, gr = X[order], y[order], groups[order]

        gkf = GroupKFold(n_splits=n_splits)
        for train_idx, val_idx in gkf.split(Xr, yr, gr):
            X_train, X_val = Xr[train_idx], Xr[val_idx]
            y_train, y_val = yr[train_idx], yr[val_idx]

            # Standardize X and y within fold
            xs = StandardScaler()
            ys = StandardScaler()
            X_train_s = xs.fit_transform(X_train)
            X_val_s = xs.transform(X_val)
            y_train_s = ys.fit_transform(y_train.reshape(-1, 1)).ravel()

            best_rmse = np.inf
            best_pred = None
            best_a = None
            best_l = None

            # Grid search on this fold
            for a in alphas:
                for l in lambdas:
                    model = ElasticNet(alpha=l, l1_ratio=a, max_iter=5000, random_state=RANDOM_SEED)
                    model.fit(X_train_s, y_train_s)
                    y_pred_s = model.predict(X_val_s)
                    y_pred = ys.inverse_transform(y_pred_s.reshape(-1, 1)).ravel()

                    rmse = math.sqrt(mean_squared_error(y_val, y_pred))
                    if rmse < best_rmse:
                        best_rmse = rmse
                        best_pred = y_pred
                        best_a = a
                        best_l = l

            all_preds.extend(best_pred)
            all_true.extend(y_val)
            if best_a is not None and best_l is not None:
                best_l1s.append(best_a)
                best_alphas.append(best_l)

    r2 = r2_score(all_true, all_preds)
    rmse = math.sqrt(mean_squared_error(all_true, all_preds))
    # Summarize best params across folds (mode + mean)
    def _mode(vals):
        if not vals:
            return None
        return max(set(vals), key=vals.count)
    best_alpha_mode = _mode(best_alphas)
    best_l1_mode = _mode(best_l1s)
    best_alpha_mean = float(np.mean(best_alphas)) if best_alphas else None
    best_l1_mean = float(np.mean(best_l1s)) if best_l1s else None
    return r2, rmse, len(sub), n_before, n_after, best_alpha_mode, best_l1_mode, best_alpha_mean, best_l1_mean


In [8]:
# Run CV for all combinations and both targets
results = []

for target_col in ["FARS1", "SARA1"]:
    print("\n=== Target:", target_col, "===")
    for combo in combinations:
        if combo["skip"]:
            results.append({
                "target": target_col,
                "combination": combo["name"],
                "r2": np.nan,
                "rmse": np.nan,
                "n_rows": 0,
                "status": "skipped",
            })
            continue
        print("Running", target_col, combo["name"], "| features:", sum(len(d) for d in combo["domains"]))
        r2, rmse, n_rows, n_before, n_after, best_alpha_mode, best_l1_mode, best_alpha_mean, best_l1_mean = run_cv_for_combination(merged_long, target_col, combo)
        if n_before - n_after == 0:
            print("Outlier removal for %s / %s: removed 0 (none), left %d" % (combo["name"], target_col, n_after))
        else:
            print("Outlier removal for %s / %s: removed %d, left %d" % (combo["name"], target_col, n_before - n_after, n_after))
        results.append({
            "target": target_col,
            "combination": combo["name"],
            "r2": r2,
            "rmse": rmse,
            "lambda_mean": best_l1_mean,
            "alpha_mean": best_alpha_mean,
            "lambda_mode": best_l1_mode,
            "alpha_mode": best_alpha_mode,
            "n_rows": n_rows,
            "status": "completed",
        })

cv_df = pd.DataFrame(results)

# Show metrics + hyperparameters per combination
cols = [c for c in ["target","combination","r2","rmse","n_rows","lambda_mode","alpha_mode","lambda_mean","alpha_mean"] if c in cv_df.columns]
cv_df[cols]



=== Target: FARS1 ===
Running FARS1 background_only | features: 8
Outlier removal for background_only / FARS1: removed 1, left 25
Running FARS1 structural_only | features: 11
Outlier removal for structural_only / FARS1: removed 0 (none), left 26
Running FARS1 diffusion_only | features: 12
Outlier removal for diffusion_only / FARS1: removed 1, left 25
Running FARS1 all_neuroimaging | features: 23
Outlier removal for all_neuroimaging / FARS1: removed 1, left 25
Running FARS1 background_structural | features: 19
Outlier removal for background_structural / FARS1: removed 1, left 25
Running FARS1 background_diffusion | features: 20
Outlier removal for background_diffusion / FARS1: removed 2, left 24
Running FARS1 background_structural_diffusion | features: 31
Outlier removal for background_structural_diffusion / FARS1: removed 2, left 24

=== Target: SARA1 ===
Running SARA1 background_only | features: 8
Outlier removal for background_only / SARA1: removed 1, left 25
Running SARA1 structura

,target,combination,r2,rmse,n_rows,lambda_mode,alpha_mode,lambda_mean,alpha_mean
0,FARS1,background_only,0.427591,19.104692,25,0.0,2.0,0.000000,1.333333
1,FARS1,structural_only,0.230665,22.677239,26,0.0,0.5,0.166667,1.000000
2,FARS1,diffusion_only,-0.001517,25.104070,25,0.5,0.5,0.333333,0.333333
3,FARS1,qsm_only,NaN,NaN,0,NaN,NaN,NaN,NaN
4,FARS1,all_neuroimaging,0.199423,22.444852,25,0.5,0.5,0.500000,0.500000
5,FARS1,background_structural,0.334065,20.606438,25,0.0,2.0,0.000000,1.500000
6,FARS1,background_diffusion,0.477802,17.707779,24,0.0,0.5,0.000000,0.666667
7,FARS1,background_qsm,NaN,NaN,0,NaN,NaN,NaN,NaN
8,FARS1,background_structural_diffusion,0.363938,19.543224,24,0.0,0.5,0.166667,1.000000
9,FARS1,background_all_neuroimaging,NaN,NaN,0,NaN,NaN,NaN,NaN


In [9]:
# Save CV results
results_dir = Path("results")
results_dir.mkdir(parents=True, exist_ok=True)

cv_df[cv_df.target == "FARS"].to_csv(results_dir / "cv_performance_fars.csv", index=False)
cv_df[cv_df.target == "SARA"].to_csv(results_dir / "cv_performance_sara.csv", index=False)
print("Saved CV CSVs")


Saved CV CSVs


## Section 5 — Variable Importance (Bootstrap)
201 bootstraps at subject level; coefficients are standardised.


In [10]:
def bootstrap_importance(df, target_col, combo, n_boot=201):
    feature_cols = []
    for domain in combo["domains"]:
        feature_cols.extend(domain)

    sub = df.dropna(subset=feature_cols + [target_col]).copy()
    sub = sub[~tukey_outliers_mask(sub, feature_cols, k=3.0)].copy()

    group_col = "subject" if "subject" in sub.columns else ("melb_id" if "melb_id" in sub.columns else ("ID" if "ID" in sub.columns else None))
    if group_col is None:
        raise KeyError("No subject/group id column found for bootstrap")
    groups = sub[group_col].values
    unique_groups = np.unique(groups)

    X = sub[feature_cols].values
    y = sub[target_col].values

    coef_list = []

    for _ in range(n_boot):
        sampled_groups = np.random.choice(unique_groups, size=len(unique_groups), replace=True)
        mask = np.isin(groups, sampled_groups)
        Xb, yb = X[mask], y[mask]

        xs = StandardScaler()
        ys = StandardScaler()
        Xs = xs.fit_transform(Xb)
        ys_ = ys.fit_transform(yb.reshape(-1, 1)).ravel()

        # Ridge-like setting (l1_ratio=0) as in the paper's best model
        model = ElasticNet(alpha=1.3, l1_ratio=0.0, max_iter=5000, random_state=RANDOM_SEED)
        model.fit(Xs, ys_)
        coef_list.append(model.coef_)

    coef_arr = np.vstack(coef_list)
    summary = {
        "feature": feature_cols,
        "coef_mean": coef_arr.mean(axis=0),
        "coef_p2_5": np.percentile(coef_arr, 2.5, axis=0),
        "coef_p97_5": np.percentile(coef_arr, 97.5, axis=0),
        "pct_nonzero": (coef_arr != 0).mean(axis=0),
    }
    return pd.DataFrame(summary)


In [11]:
# Run bootstrap for primary combination only (background_structural_diffusion)
primary = [c for c in combinations if c["name"] == "background_structural_diffusion"][0]

importance_dir = Path("results/variable_importance")
importance_dir.mkdir(parents=True, exist_ok=True)

for target_col in ["FARS1", "SARA1"]:
    print("Bootstrap importance:", target_col)
    imp = bootstrap_importance(merged_long, target_col, primary, n_boot=201)
    imp.to_csv(importance_dir / f"importance_{target_col.lower()}_{primary['name']}.csv", index=False)


Bootstrap importance: FARS1
Bootstrap importance: SARA1


## Section 6 — Composite + Sensitivity
This follows the paper's composite construction and paired v1/v2 sensitivity analysis.


In [12]:
def compute_composite(df, target_col, combo, alpha=1.3, l1_ratio=0.0):
    feature_cols = []
    for domain in combo["domains"]:
        feature_cols.extend(domain)

    sub = df.dropna(subset=feature_cols + [target_col]).copy()
    sub = sub[~tukey_outliers_mask(sub, feature_cols, k=3.0)].copy()

    X = sub[feature_cols].values
    y = sub[target_col].values

    xs = StandardScaler()
    ys = StandardScaler()
    Xs = xs.fit_transform(X)
    ys_ = ys.fit_transform(y.reshape(-1, 1)).ravel()

    model = ElasticNet(alpha=alpha, l1_ratio=l1_ratio, max_iter=5000, random_state=RANDOM_SEED)
    model.fit(Xs, ys_)

    # Convert standardised coefficients to raw scale
    coef_std = model.coef_
    feature_sd = xs.scale_
    coef_raw = coef_std / feature_sd

    composite = (np.abs(coef_raw) * X).sum(axis=1)
    sub = sub.copy()
    sub["composite"] = composite
    return sub


In [13]:
# Composite + sensitivity with subject-level LOO (OOF predictions)
primary = [c for c in combinations if c["name"] == "background_structural_diffusion"][0]

# Build v1/v2 feature lists
feature_cols_v1 = [f for domain in primary["domains"] for f in domain]
feature_cols_v2 = [f.replace("_v1", "_v2") if f.endswith("_v1") else f for f in feature_cols_v1]
base_cols = [f.replace("_v1", "") for f in feature_cols_v1]

# Identify subject id column
id_col = "subject" if "subject" in merged_long.columns else ("melb_id" if "melb_id" in merged_long.columns else ("ID" if "ID" in merged_long.columns else None))
if id_col is None:
    raise KeyError("No subject id column found for composite")

# Build long format (v1 + v2)

def build_long_format(df):
    df_v1 = df[[id_col] + feature_cols_v1 + ["FARS1"]].copy()
    df_v1.columns = [id_col] + base_cols + ["FARS"]
    df_v1["visit"] = 1

    df_v2 = df[[id_col] + feature_cols_v2 + ["FARS2"]].copy()
    df_v2.columns = [id_col] + base_cols + ["FARS"]
    df_v2["visit"] = 2

    return pd.concat([df_v1, df_v2], ignore_index=True)

long_df = build_long_format(merged_long)

# Subject-level Leave-One-Out composite (Out-Of-Fold predictions)

def compute_composite_loocv(df_long, feature_cols, target_col, subject_col, alpha=1.3, l1_ratio=0.0):
    sub = df_long.dropna(subset=feature_cols + [target_col]).copy()
    sub = sub[~tukey_outliers_mask(sub, feature_cols, k=3.0)].copy()

    subjects = sub[subject_col].unique()
    results = []

    for subj in subjects:
        test_mask = sub[subject_col] == subj
        train_mask = ~test_mask

        X_train = sub.loc[train_mask, feature_cols].values
        y_train = sub.loc[train_mask, target_col].values
        X_test = sub.loc[test_mask, feature_cols].values

        xs = StandardScaler()
        ys = StandardScaler()
        X_train_s = xs.fit_transform(X_train)
        y_train_s = ys.fit_transform(y_train.reshape(-1, 1)).ravel()
        X_test_s = xs.transform(X_test)

        model = ElasticNet(alpha=alpha, l1_ratio=l1_ratio, max_iter=5000, random_state=RANDOM_SEED)
        model.fit(X_train_s, y_train_s)

        y_pred_s = model.predict(X_test_s)
        y_pred = ys.inverse_transform(y_pred_s.reshape(-1, 1)).ravel()

        test_rows = sub.loc[test_mask].copy()
        test_rows["composite_pred"] = y_pred
        results.append(test_rows)

    return pd.concat(results, ignore_index=True)

result = compute_composite_loocv(long_df, base_cols, "FARS", id_col, alpha=1.3, l1_ratio=0.0)

# Pair v1/v2 predictions
v1_pred = result[result["visit"] == 1][[id_col, "composite_pred"]]
v2_pred = result[result["visit"] == 2][[id_col, "composite_pred"]]
paired = v1_pred.merge(v2_pred, on=id_col, suffixes=("_v1", "_v2")).dropna()

# Sensitivity metrics
diff = paired["composite_pred_v2"] - paired["composite_pred_v1"]
cohens_d = diff.mean() / diff.std(ddof=1)

from scipy.stats import ttest_rel
p_val = ttest_rel(paired["composite_pred_v2"], paired["composite_pred_v1"]).pvalue

print(f"Composite Cohen's d: {cohens_d:.3f}")
print(f"Paired t p-value:    {p_val:.4f}")

# Sensitivity summary table (Composite + FARS baseline)
mean_change = diff.mean()
sd_change = diff.std(ddof=1)

if "FARS1" in merged_long.columns and "FARS2" in merged_long.columns:
    fars_pair = merged_long[["FARS1", "FARS2"]].dropna()
    fars_diff = fars_pair["FARS2"] - fars_pair["FARS1"]
    fars_mean = fars_diff.mean()
    fars_sd = fars_diff.std(ddof=1)
    fars_d = fars_mean / fars_sd if fars_sd != 0 else float("nan")
else:
    fars_mean = fars_sd = fars_d = float("nan")

sensitivity_table = pd.DataFrame([
    {"biomarker": "Composite", "n": len(diff), "mean_change": mean_change, "sd_change": sd_change, "d_score": cohens_d},
    {"biomarker": "FARS", "n": len(fars_diff) if 'fars_diff' in locals() else 0, "mean_change": fars_mean, "sd_change": fars_sd, "d_score": fars_d},
])

print("Sensitivity table:")
print(sensitivity_table)


Composite Cohen's d: 0.338
Paired t p-value:    0.1116
Sensitivity table:
   biomarker   n  mean_change  sd_change   d_score
0  Composite  24     1.477595   4.375007  0.337736
1       FARS  26     6.819231   7.938615  0.858995


In [14]:
# Save sensitivity summary
sens = pd.DataFrame({
    "metric": ["cohens_d", "paired_t_p"],
    "value": [cohens_d, p_val],
})

sens.to_csv(Path("results") / "composite_sensitivity.csv", index=False)
print("Saved composite_sensitivity.csv")


Saved composite_sensitivity.csv


## DEMO2, v1 + v2

In [15]:
# --- fix: map target to FARS1/SARA1 if FARS/SARA not present ---
if 'FARS' not in merged_long.columns and 'FARS1' in merged_long.columns:
    merged_long['FARS'] = merged_long['FARS1']
    print('Demo2 fix: set FARS = FARS1 for CV demo')

if 'SARA' not in merged_long.columns and 'SARA1' in merged_long.columns:
    merged_long['SARA'] = merged_long['SARA1']
    print('Demo2 fix: set SARA = SARA1 for CV demo')

Demo2 fix: set FARS = FARS1 for CV demo
Demo2 fix: set SARA = SARA1 for CV demo


In [16]:
# --- CV ---

def run_cv_demo2(df, target_col, combo, subject_col, alphas=None, l1_ratios=None):
    if alphas is None:
        alphas = np.logspace(-2, 2, 10)
    if l1_ratios is None:
        l1_ratios = [0.1, 0.5, 0.9]

    feature_cols = [f for domain in combo["domains"] for f in domain]
    sub = df.dropna(subset=feature_cols + [target_col]).copy()
    n_before = len(sub)

    outlier_mask = tukey_outliers_mask(sub, feature_cols, k=3.0)
    sub = sub[~outlier_mask].copy()
    n_after = len(sub)

    if n_before - n_after == 0:
        print("Outlier removal for %s / %s: removed 0 (none), left %d" % (combo["name"], target_col, n_after))
    else:
        print("Outlier removal for %s / %s: removed %d, left %d" % (combo["name"], target_col, n_before - n_after, n_after))

    groups = sub[subject_col].values
    X = sub[feature_cols].values
    y = sub[target_col].values

    subjects = np.unique(groups)
    all_preds, all_true = [], []
    best_alphas, best_l1s = [], []

    for subj in subjects:
        test_mask = groups == subj
        train_mask = ~test_mask

        X_train = X[train_mask]
        y_train = y[train_mask]
        X_test = X[test_mask]
        y_test = y[test_mask]

        xs = StandardScaler()
        X_train_s = xs.fit_transform(X_train)
        X_test_s = xs.transform(X_test)

        best_rmse, best_pred = np.inf, None
        best_a, best_l = None, None
        for a in alphas:
            for l in l1_ratios:
                m = ElasticNet(alpha=a, l1_ratio=l, max_iter=5000, random_state=RANDOM_SEED)
                m.fit(X_train_s, y_train)
                pred = m.predict(X_test_s)
                rmse = np.sqrt(mean_squared_error(y_test, pred))
                if rmse < best_rmse:
                    best_rmse = rmse
                    best_pred = pred
                    best_a = a
                    best_l = l

        all_preds.extend(best_pred)
        all_true.extend(y_test)
        if best_a is not None and best_l is not None:
            best_alphas.append(best_a)
            best_l1s.append(best_l)

    r2 = r2_score(all_true, all_preds)
    rmse = np.sqrt(mean_squared_error(all_true, all_preds))

    def _mode(vals):
        if not vals:
            return None
        return max(set(vals), key=vals.count)

    return {
        "r2": r2,
        "rmse": rmse,
        "n_rows": len(sub),
        "lambda_mode": _mode(best_alphas),
        "alpha_mode": _mode(best_l1s),
        "lambda_mean": float(np.mean(best_alphas)) if best_alphas else None,
        "alpha_mean": float(np.mean(best_l1s)) if best_l1s else None,
    }

# Detect subject column
subject_col_demo2 = "subject" if "subject" in merged_long.columns else ("melb_id" if "melb_id" in merged_long.columns else ("ID" if "ID" in merged_long.columns else None))
if subject_col_demo2 is None:
    raise KeyError("No subject id column found for demo2 CV")

results_demo2 = []
for target_col in ["FARS"]:
    for combo in combinations:
        if combo.get("skip"):
            results_demo2.append({
                "target": target_col,
                "combination": combo["name"],
                "r2": np.nan,
                "rmse": np.nan,
                "n_rows": 0,
                "lambda_mode": None,
                "alpha_mode": None,
                "lambda_mean": None,
                "alpha_mean": None,
            })
            continue
        print("Running", target_col, combo["name"], "| features:", sum(len(d) for d in combo["domains"]))
        res = run_cv_demo2(merged_long, target_col, combo, subject_col_demo2)
        res.update({"target": target_col, "combination": combo["name"]})
        results_demo2.append(res)

cv_df_demo2 = pd.DataFrame(results_demo2)
cols = [c for c in ["target","combination","r2","rmse","n_rows","lambda_mode","alpha_mode","lambda_mean","alpha_mean"] if c in cv_df_demo2.columns]
cv_df_demo2[cols]


Running FARS background_only | features: 8
Outlier removal for background_only / FARS: removed 1, left 25
Running FARS structural_only | features: 11
Outlier removal for structural_only / FARS: removed 0 (none), left 26
Running FARS diffusion_only | features: 12
Outlier removal for diffusion_only / FARS: removed 1, left 25
Running FARS all_neuroimaging | features: 23
Outlier removal for all_neuroimaging / FARS: removed 1, left 25
Running FARS background_structural | features: 19
Outlier removal for background_structural / FARS: removed 1, left 25
Running FARS background_diffusion | features: 20
Outlier removal for background_diffusion / FARS: removed 2, left 24
Running FARS background_structural_diffusion | features: 31
Outlier removal for background_structural_diffusion / FARS: removed 2, left 24


,target,combination,r2,rmse,n_rows,lambda_mode,alpha_mode,lambda_mean,alpha_mean
0,FARS,background_only,0.639670,15.157836,25,4.641589,0.5,8.268306,0.596000
1,FARS,structural_only,0.569120,16.971123,26,0.010000,0.5,7.844289,0.576923
2,FARS,diffusion_only,0.690421,13.957272,25,0.010000,0.9,8.807192,0.676000
3,FARS,qsm_only,NaN,NaN,0,NaN,NaN,NaN,NaN
4,FARS,all_neuroimaging,0.816382,10.749113,25,0.010000,0.1,6.695558,0.452000
5,FARS,background_structural,0.714776,13.485913,25,4.641589,0.1,6.705810,0.356000
6,FARS,background_diffusion,0.789498,11.242794,24,1.668101,0.9,4.646816,0.533333
7,FARS,background_qsm,NaN,NaN,0,NaN,NaN,NaN,NaN
8,FARS,background_structural_diffusion,0.779835,11.497956,24,1.668101,0.1,4.393549,0.366667
9,FARS,background_all_neuroimaging,NaN,NaN,0,NaN,NaN,NaN,NaN


In [17]:
# --- Demo2 CV for SARA ---
results_demo2_sara = []
for target_col in ["SARA"]:
    for combo in combinations:
        if combo.get("skip"):
            results_demo2_sara.append({
                "target": target_col,
                "combination": combo["name"],
                "r2": np.nan,
                "rmse": np.nan,
                "n_rows": 0,
                "lambda_mode": None,
                "alpha_mode": None,
                "lambda_mean": None,
                "alpha_mean": None,
            })
            continue
        print("Running", target_col, combo["name"], "| features:", sum(len(d) for d in combo["domains"]))
        res = run_cv_demo2(merged_long, target_col, combo, subject_col_demo2)
        res.update({"target": target_col, "combination": combo["name"]})
        results_demo2_sara.append(res)

cv_df_demo2_sara = pd.DataFrame(results_demo2_sara)
cols = [c for c in ["target","combination","r2","rmse","n_rows","lambda_mode","alpha_mode","lambda_mean","alpha_mean"] if c in cv_df_demo2_sara.columns]
cv_df_demo2_sara[cols]


Running SARA background_only | features: 8
Outlier removal for background_only / SARA: removed 1, left 25
Running SARA structural_only | features: 11
Outlier removal for structural_only / SARA: removed 0 (none), left 26
Running SARA diffusion_only | features: 12
Outlier removal for diffusion_only / SARA: removed 1, left 25
Running SARA all_neuroimaging | features: 23
Outlier removal for all_neuroimaging / SARA: removed 1, left 25
Running SARA background_structural | features: 19
Outlier removal for background_structural / SARA: removed 1, left 25
Running SARA background_diffusion | features: 20
Outlier removal for background_diffusion / SARA: removed 2, left 24
Running SARA background_structural_diffusion | features: 31
Outlier removal for background_structural_diffusion / SARA: removed 2, left 24


,target,combination,r2,rmse,n_rows,lambda_mode,alpha_mode,lambda_mean,alpha_mean
0,SARA,background_only,0.546777,5.125593,25,1.668101,0.5,4.979103,0.564000
1,SARA,structural_only,0.627140,4.787067,26,0.010000,0.9,2.490751,0.592308
2,SARA,diffusion_only,0.562723,5.144318,25,0.010000,0.9,1.652523,0.660000
3,SARA,qsm_only,NaN,NaN,0,NaN,NaN,NaN,NaN
4,SARA,all_neuroimaging,0.807460,3.413580,25,0.010000,0.1,1.706562,0.468000
5,SARA,background_structural,0.686955,4.259823,25,0.010000,0.9,3.193407,0.564000
6,SARA,background_diffusion,0.608200,4.734082,24,1.668101,0.9,4.432708,0.566667
7,SARA,background_qsm,NaN,NaN,0,NaN,NaN,NaN,NaN
8,SARA,background_structural_diffusion,0.667845,4.358867,24,0.077426,0.9,4.288005,0.583333
9,SARA,background_all_neuroimaging,NaN,NaN,0,NaN,NaN,NaN,NaN


In [18]:
# --- prep: ensure merged_long is wide with FARS1/FARS2 ---
melb_path_demo2 = Path('data/processed/melbourne_frda_merged.csv')
if 'FARS1' not in merged_long.columns:
    merged_long = pd.read_csv(melb_path_demo2)
    print('Demo2 prep: reloaded wide melbourne data for FARS1/FARS2 blocks')


In [19]:
# --- fix: rebuild feature_cols_v1/v2 from wide columns ---
if 'FARS1' in merged_long.columns:
    # Use actual _v1/_v2 columns present in merged_long
    feature_cols_v1 = [c for c in merged_long.columns if c.endswith('_v1')]
    feature_cols_v2 = [c for c in merged_long.columns if c.endswith('_v2')]
    base_cols = [c.replace('_v1','') for c in feature_cols_v1]


In [20]:
# --- Demo2 Composite Sensitivity: FARS ---
primary = [c for c in combinations if c["name"] == "background_structural_diffusion"][0]
feature_cols_v1 = [f for domain in primary["domains"] for f in domain]
feature_cols_v2 = [f.replace("_v1", "_v2") if f.endswith("_v1") else f for f in feature_cols_v1]
base_cols = [f.replace("_v1", "") for f in feature_cols_v1]

id_col = "subject" if "subject" in merged_long.columns else ("melb_id" if "melb_id" in merged_long.columns else ("ID" if "ID" in merged_long.columns else None))
if id_col is None:
    raise KeyError("No subject id column found for composite")

# Build long format for FARS

def build_long_format_fars(df):
    df_v1 = df[[id_col] + feature_cols_v1 + ["FARS1"]].copy()
    df_v1.columns = [id_col] + base_cols + ["FARS"]
    df_v1["visit"] = 1
    df_v2 = df[[id_col] + feature_cols_v2 + ["FARS2"]].copy()
    df_v2.columns = [id_col] + base_cols + ["FARS"]
    df_v2["visit"] = 2
    return pd.concat([df_v1, df_v2], ignore_index=True)

long_fars = build_long_format_fars(merged_long)

# Out-Of-Fold composite
result_fars = compute_composite_loocv(long_fars, base_cols, "FARS", id_col, alpha=1.3, l1_ratio=0.0)

v1_pred = result_fars[result_fars["visit"] == 1][[id_col, "composite_pred"]]
v2_pred = result_fars[result_fars["visit"] == 2][[id_col, "composite_pred"]]
paired = v1_pred.merge(v2_pred, on=id_col, suffixes=("_v1", "_v2")).dropna()

diff = paired["composite_pred_v2"] - paired["composite_pred_v1"]
cohens_d = diff.mean() / diff.std(ddof=1)
from scipy.stats import ttest_rel
p_val = ttest_rel(paired["composite_pred_v2"], paired["composite_pred_v1"]).pvalue

# FARS baseline
fars_pair = merged_long[["FARS1", "FARS2"]].dropna()
fars_diff = fars_pair["FARS2"] - fars_pair["FARS1"]

sensitivity_fars = pd.DataFrame([
    {"biomarker": "Composite", "n": len(diff), "mean_change": diff.mean(), "sd_change": diff.std(ddof=1), "d_score": cohens_d, "p_value": p_val},
    {"biomarker": "FARS", "n": len(fars_diff), "mean_change": fars_diff.mean(), "sd_change": fars_diff.std(ddof=1), "d_score": fars_diff.mean()/fars_diff.std(ddof=1), "p_value": np.nan},
])

print("Sensitivity (FARS):")
print(sensitivity_fars)


Sensitivity (FARS):
   biomarker   n  mean_change  sd_change   d_score   p_value
0  Composite  24     1.477595   4.375007  0.337736  0.111598
1       FARS  26     6.819231   7.938615  0.858995       NaN


In [21]:
# --- Demo2 Composite Sensitivity: SARA ---
# Build long format for SARA

def build_long_format_sara(df):
    df_v1 = df[[id_col] + feature_cols_v1 + ["SARA1"]].copy()
    df_v1.columns = [id_col] + base_cols + ["SARA"]
    df_v1["visit"] = 1
    df_v2 = df[[id_col] + feature_cols_v2 + ["SARA2"]].copy()
    df_v2.columns = [id_col] + base_cols + ["SARA"]
    df_v2["visit"] = 2
    return pd.concat([df_v1, df_v2], ignore_index=True)

long_sara = build_long_format_sara(merged_long)

result_sara = compute_composite_loocv(long_sara, base_cols, "SARA", id_col, alpha=1.3, l1_ratio=0.0)

v1_pred = result_sara[result_sara["visit"] == 1][[id_col, "composite_pred"]]
v2_pred = result_sara[result_sara["visit"] == 2][[id_col, "composite_pred"]]
paired = v1_pred.merge(v2_pred, on=id_col, suffixes=("_v1", "_v2")).dropna()

diff = paired["composite_pred_v2"] - paired["composite_pred_v1"]
cohens_d = diff.mean() / diff.std(ddof=1)
from scipy.stats import ttest_rel
p_val = ttest_rel(paired["composite_pred_v2"], paired["composite_pred_v1"]).pvalue

# SARA baseline
sara_pair = merged_long[["SARA1", "SARA2"]].dropna()
sara_diff = sara_pair["SARA2"] - sara_pair["SARA1"]

sensitivity_sara = pd.DataFrame([
    {"biomarker": "Composite", "n": len(diff), "mean_change": diff.mean(), "sd_change": diff.std(ddof=1), "d_score": cohens_d, "p_value": p_val},
    {"biomarker": "SARA", "n": len(sara_diff), "mean_change": sara_diff.mean(), "sd_change": sara_diff.std(ddof=1), "d_score": sara_diff.mean()/sara_diff.std(ddof=1), "p_value": np.nan},
])

print("Sensitivity (SARA):")
print(sensitivity_sara)


Sensitivity (SARA):
   biomarker   n  mean_change  sd_change   d_score   p_value
0  Composite  21     0.677589   1.466473  0.462054  0.046952
1       SARA  23     2.173913   2.806683  0.774549       NaN


## Aligned Pipeline (Full Protocol)
This section implements the Week2 + Pipeline plan without relying on earlier demo blocks.


In [22]:
# Load wide + build long format (Melbourne only)
melb_path = Path('data/processed/melbourne_frda_merged.csv')
melb_wide = pd.read_csv(melb_path)

subject_col = 'subject' if 'subject' in melb_wide.columns else ('melb_id' if 'melb_id' in melb_wide.columns else 'ID')

# Feature definitions (baseline + structural_ext)
# Wide (visit-specific) background columns available in melb_wide
background_wide = [
    'age1',
    'age2',
    'age_at_onset',
    'dur1',
    'dur2',
    'sex',
    'GAA1',
    'GAA2',
]

# Long-format background used for modeling (per-row visit-aligned age/duration)
background = [
    'age',
    'age_at_onset',
    'dur',
    'sex',
    'GAA1',
    'GAA2',
]

structural = ['SCP','MCP','ICP','Midbrain','Pons','Medulla','AntCBLM','SupPostCBLM','InfPostCBLM','FlocCBLM','VermisCBLM']
structural_ext = structural + ['CSA_C1','CSA_C2','ECC_C1','ECC_C2']

diffusion = ['FASCP','FAMCP','FAICP','MDSCP','MDMCP','MDICP','ADSCP','ADMCP','ADICP','RDSCP','RDMCP','RDICP']

# Combinations (flat features)
combinations = [
    {"name":"background_only", "features": background},
    {"name":"structural_only", "features": structural},
    {"name":"diffusion_only", "features": diffusion},
    {"name":"all_neuroimaging", "features": structural + diffusion},
    {"name":"background_structural", "features": background + structural},
    {"name":"background_diffusion", "features": background + diffusion},
    {"name":"background_structural_diffusion", "features": background + structural + diffusion},
    # Extended
    {"name":"structural_ext_only", "features": structural_ext},
    {"name":"background_structural_ext", "features": background + structural_ext},
    {"name":"background_structural_ext_diffusion", "features": background + structural_ext + diffusion},
]

# Build long format from v1/v2
v1_cols = [c for c in melb_wide.columns if c.endswith('_v1')]
v2_cols = [c for c in melb_wide.columns if c.endswith('_v2')]
base_cols = sorted({c[:-3] for c in v1_cols} & {c[:-3] for c in v2_cols})

rows = []
for _, row in melb_wide.iterrows():
    subj = row[subject_col]
    # visit1
    r1 = {subject_col: subj, 'visit': 1}
    for b in base_cols:
        r1[b] = row.get(b+'_v1')
    r1['FARS'] = row.get('FARS1')
    r1['SARA'] = row.get('SARA1')
    r1['age'] = row.get('age1')
    r1['dur'] = row.get('dur1')
    r1['age_at_onset'] = row.get('age_at_onset')
    r1['sex'] = row.get('sex')
    r1['GAA1'] = row.get('GAA1')
    r1['GAA2'] = row.get('GAA2')
    rows.append(r1)
    # visit2
    r2 = {subject_col: subj, 'visit': 2}
    for b in base_cols:
        r2[b] = row.get(b+'_v2')
    r2['FARS'] = row.get('FARS2')
    r2['SARA'] = row.get('SARA2')
    r2['age'] = row.get('age2')
    r2['dur'] = row.get('dur2')
    r2['age_at_onset'] = row.get('age_at_onset')
    r2['sex'] = row.get('sex')
    r2['GAA1'] = row.get('GAA1')
    r2['GAA2'] = row.get('GAA2')
    rows.append(r2)

merged_long = pd.DataFrame(rows)

print('Long format:', merged_long.shape)


Long format: (52, 37)


In [23]:
# Helpers: d_score and LOO CV with coef storage

def compute_cohens_d_from_oof(oof_df, subject_col='subject', pred_col='pred', visit_col='visit'):
    paired = (oof_df.sort_values([subject_col, visit_col])
              .groupby(subject_col)[pred_col]
              .apply(list))
    paired = paired[paired.map(len) == 2]
    if len(paired) < 2:
        return np.nan
    diffs = paired.map(lambda x: x[1] - x[0])
    return diffs.mean() / diffs.std(ddof=1)


def run_cv_loocv_with_coefs(df_long, feature_cols, target_col, subject_col,
                             l1_ratios=None, reg_strengths=None):
    if l1_ratios is None:
        l1_ratios = np.linspace(0, 1, 11)
    if reg_strengths is None:
        reg_strengths = np.linspace(0, 20, 201)

    sub = df_long.dropna(subset=feature_cols + [target_col]).copy()
    sub = sub[~tukey_outliers_mask(sub, feature_cols, k=3.0)].copy()

    groups = sub[subject_col].values
    X = sub[feature_cols].values
    y = sub[target_col].values

    n_subjects = len(np.unique(groups))
    gkf = GroupKFold(n_splits=n_subjects)

    all_preds, all_true = [], []
    oof_rows = []
    coef_rows = []

    for train_idx, test_idx in gkf.split(X, y, groups):
        X_train, X_test = X[train_idx], X[test_idx]
        y_train, y_test = y[train_idx], y[test_idx]
        group_test = groups[test_idx]

        xs = StandardScaler()
        X_train_s = xs.fit_transform(X_train)
        X_test_s = xs.transform(X_test)

        best_rmse, best_pred, best_a, best_l = np.inf, None, None, None
        for a in reg_strengths:
            for l in l1_ratios:
                m = ElasticNet(alpha=a, l1_ratio=l, max_iter=5000, random_state=RANDOM_SEED)
                m.fit(X_train_s, y_train)
                pred = m.predict(X_test_s)
                rmse = np.sqrt(mean_squared_error(y_test, pred))
                if rmse < best_rmse:
                    best_rmse, best_pred, best_a, best_l = rmse, pred, a, l
                    best_model = m

        all_preds.extend(best_pred)
        all_true.extend(y_test)

        # store OOF predictions
        for sid, v, p in zip(group_test, sub.iloc[test_idx]['visit'], best_pred):
            oof_rows.append({subject_col: sid, 'visit': v, 'pred': p})

        # store coef for feature selection
        coef_rows.append(best_model.coef_)

    r2 = r2_score(all_true, all_preds)
    rmse = np.sqrt(mean_squared_error(all_true, all_preds))
    oof_df = pd.DataFrame(oof_rows)
    coef_mat = np.vstack(coef_rows) if coef_rows else np.zeros((0, len(feature_cols)))
    return r2, rmse, oof_df, coef_mat


# --- New: inner CV hyperparameter selection by Cohen's d ---
def select_params_by_inner_cv_d(
    df_long_train,
    feature_cols,
    target_col,
    subject_col,
    *,
    inner_folds=5,
    l1_ratios=None,
    alphas=None,
):
    if l1_ratios is None:
        l1_ratios = [0.1, 0.5, 0.9]
    if alphas is None:
        alphas = np.logspace(-2, 2, 10)

    sub = df_long_train.dropna(subset=list(feature_cols) + [target_col]).copy()
    sub = sub[~tukey_outliers_mask(sub, feature_cols, k=3.0)].copy()
    if len(sub) == 0:
        return None, None, pd.DataFrame(), np.nan, np.nan

    groups = sub[subject_col].values
    n_unique = len(np.unique(groups))
    n_splits = min(int(inner_folds), int(n_unique))
    if n_splits < 2:
        return None, None, pd.DataFrame(), np.nan, np.nan

    X = sub[list(feature_cols)].values
    y = sub[target_col].values

    best_d = -np.inf
    best_rmse = np.inf
    best_alpha = None
    best_l1 = None
    best_oof_df = pd.DataFrame()

    gkf = GroupKFold(n_splits=n_splits)
    for alpha in alphas:
        for l1 in l1_ratios:
            oof_rows = []
            for train_idx, val_idx in gkf.split(X, y, groups):
                X_train, X_val = X[train_idx], X[val_idx]
                y_train, y_val = y[train_idx], y[val_idx]
                groups_val = groups[val_idx]
                visits_val = sub.iloc[val_idx]['visit'].values

                xs = StandardScaler()
                X_train_s = xs.fit_transform(X_train)
                X_val_s = xs.transform(X_val)

                m = ElasticNet(alpha=float(alpha), l1_ratio=float(l1), max_iter=5000, random_state=RANDOM_SEED)
                m.fit(X_train_s, y_train)
                pred = m.predict(X_val_s)

                for sid, v, p, t in zip(groups_val, visits_val, pred, y_val):
                    oof_rows.append({subject_col: sid, 'visit': v, 'pred': p, 'true': t})

            oof_df = pd.DataFrame(oof_rows)
            d = compute_cohens_d_from_oof(oof_df, subject_col, pred_col='pred', visit_col='visit')
            tmp = oof_df[['true', 'pred']].dropna()
            rmse = np.nan if len(tmp) == 0 else float(np.sqrt(mean_squared_error(tmp['true'], tmp['pred'])))

            d_val = -np.inf if pd.isna(d) else float(d)
            rmse_val = np.inf if pd.isna(rmse) else float(rmse)

            better = (d_val > best_d) or (
                d_val == best_d and (rmse_val < best_rmse)
            ) or (
                d_val == best_d and rmse_val == best_rmse and (best_alpha is None or float(alpha) < float(best_alpha))
            )

            if better:
                best_d = d_val
                best_rmse = rmse_val
                best_alpha = float(alpha)
                best_l1 = float(l1)
                best_oof_df = oof_df

    out_d = np.nan if best_d == -np.inf else float(best_d)
    out_rmse = np.nan if best_rmse == np.inf else float(best_rmse)
    return best_alpha, best_l1, best_oof_df, out_d, out_rmse


# --- New: outer LOO CV with d-tuned hyperparameters + coef storage ---
def run_loocv_d_tuned_with_coefs(
    df_long,
    feature_cols,
    target_col,
    subject_col,
    *,
    inner_folds=5,
    l1_ratios=None,
    alphas=None,
):
    if l1_ratios is None:
        l1_ratios = [0.1, 0.5, 0.9]
    if alphas is None:
        alphas = np.logspace(-2, 2, 10)

    sub = df_long.dropna(subset=list(feature_cols) + [target_col]).copy()
    sub = sub[~tukey_outliers_mask(sub, feature_cols, k=3.0)].copy()

    n_rows = int(len(sub))
    if n_rows == 0:
        return {
            'd_score': np.nan,
            'rmse': np.nan,
            'r2': np.nan,
            'n_rows': 0,
            'n_subjects': 0,
            'alpha_mode': None,
            'l1_ratio_mode': None,
            'alpha_mean': None,
            'l1_ratio_mean': None,
            'oof_df': pd.DataFrame(),
            'coef_mat': np.zeros((0, len(feature_cols))),
            'chosen_params_df': pd.DataFrame(),
        }

    groups = sub[subject_col].values
    subjects = np.unique(groups)
    n_subjects = int(len(subjects))

    oof_rows = []
    coef_rows = []
    chosen_params_rows = []

    X_all = sub[list(feature_cols)].values
    y_all = sub[target_col].values

    for test_subject in subjects:
        test_mask = groups == test_subject
        train_mask = ~test_mask

        train_df = sub.loc[train_mask].copy()
        best_alpha, best_l1, _, inner_d, inner_rmse = select_params_by_inner_cv_d(
            train_df,
            feature_cols,
            target_col,
            subject_col,
            inner_folds=inner_folds,
            l1_ratios=l1_ratios,
            alphas=alphas,
        )

        if best_alpha is None or best_l1 is None:
            continue

        X_train, X_test = X_all[train_mask], X_all[test_mask]
        y_train, y_test = y_all[train_mask], y_all[test_mask]
        visits_test = sub.loc[test_mask, 'visit'].values

        xs = StandardScaler()
        X_train_s = xs.fit_transform(X_train)
        X_test_s = xs.transform(X_test)

        m = ElasticNet(alpha=float(best_alpha), l1_ratio=float(best_l1), max_iter=5000, random_state=RANDOM_SEED)
        m.fit(X_train_s, y_train)
        pred = m.predict(X_test_s)

        chosen_params_rows.append({
            'test_subject': test_subject,
            'alpha': float(best_alpha),
            'l1_ratio': float(best_l1),
            'inner_d': inner_d,
            'inner_rmse': inner_rmse,
        })

        for v, p, t in zip(visits_test, pred, y_test):
            oof_rows.append({subject_col: test_subject, 'visit': v, 'pred': p, 'true': t})

        coef_rows.append(m.coef_)

    oof_df = pd.DataFrame(oof_rows)
    coef_mat = np.vstack(coef_rows) if coef_rows else np.zeros((0, len(feature_cols)))
    chosen_params_df = pd.DataFrame(chosen_params_rows)

    d_score = compute_cohens_d_from_oof(oof_df, subject_col, pred_col='pred', visit_col='visit')
    tmp = oof_df[['true', 'pred']].dropna() if len(oof_df) else pd.DataFrame(columns=['true', 'pred'])
    rmse = np.nan if len(tmp) == 0 else float(np.sqrt(mean_squared_error(tmp['true'], tmp['pred'])))
    r2 = np.nan if len(tmp) < 2 else float(r2_score(tmp['true'], tmp['pred']))

    def _mode(vals):
        vals = [v for v in vals if v is not None and np.isfinite(v)]
        if not vals:
            return None
        return max(set(vals), key=vals.count)

    alpha_mode = _mode(chosen_params_df['alpha'].tolist()) if 'alpha' in chosen_params_df.columns else None
    l1_mode = _mode(chosen_params_df['l1_ratio'].tolist()) if 'l1_ratio' in chosen_params_df.columns else None
    alpha_mean = float(np.mean(chosen_params_df['alpha'])) if 'alpha' in chosen_params_df.columns and len(chosen_params_df) else None
    l1_mean = float(np.mean(chosen_params_df['l1_ratio'])) if 'l1_ratio' in chosen_params_df.columns and len(chosen_params_df) else None

    return {
        'd_score': d_score,
        'rmse': rmse,
        'r2': r2,
        'n_rows': n_rows,
        'n_subjects': n_subjects,
        'alpha_mode': alpha_mode,
        'l1_ratio_mode': l1_mode,
        'alpha_mean': alpha_mean,
        'l1_ratio_mean': l1_mean,
        'oof_df': oof_df,
        'coef_mat': coef_mat,
        'chosen_params_df': chosen_params_df,
    }


In [24]:
# Run LOO CV for all combinations (FARS + SARA)
results = []
oof_dict = {}
coef_dict = {}

for combo in combinations:
    feats = combo['features']
    for target in ['FARS', 'SARA']:
        r2, rmse, oof_df, coef_mat = run_cv_loocv_with_coefs(merged_long, feats, target, subject_col)
        d_score = compute_cohens_d_from_oof(oof_df, subject_col, 'pred', 'visit')
        results.append({
            'combination': combo['name'],
            'n_features': len(feats),
            'target': target,
            'r2': r2,
            'rmse': rmse,
            'd_score': d_score,
        })
        oof_dict[(combo['name'], target)] = oof_df
        coef_dict[(combo['name'], target)] = coef_mat

cv_df = pd.DataFrame(results).sort_values('d_score', ascending=False)
cv_df


# --- New: Cohen's d–tuned LOO CV (nested CV) ---
results_d_tuned = []
oof_dict_d_tuned = {}
coef_dict_d_tuned = {}
params_dict_d_tuned = {}

alphas_d = np.logspace(-2, 2, 10)
l1_ratios_d = [0.1, 0.5, 0.9]
inner_folds_d = 5

for combo in combinations:
    feats = combo['features']
    for target in ['FARS', 'SARA']:
        res = run_loocv_d_tuned_with_coefs(
            merged_long,
            feats,
            target,
            subject_col,
            inner_folds=inner_folds_d,
            l1_ratios=l1_ratios_d,
            alphas=alphas_d,
        )
        results_d_tuned.append({
            'combination': combo['name'],
            'n_features': len(feats),
            'target': target,
            'd_score': res['d_score'],
            'rmse': res['rmse'],
            'r2': res['r2'],
            'n_rows': res['n_rows'],
            'n_subjects': res['n_subjects'],
            'alpha_mode': res['alpha_mode'],
            'l1_ratio_mode': res['l1_ratio_mode'],
            'alpha_mean': res['alpha_mean'],
            'l1_ratio_mean': res['l1_ratio_mean'],
        })
        oof_dict_d_tuned[(combo['name'], target)] = res['oof_df']
        coef_dict_d_tuned[(combo['name'], target)] = res['coef_mat']
        params_dict_d_tuned[(combo['name'], target)] = res['chosen_params_df']

cv_df_d_tuned = pd.DataFrame(results_d_tuned).sort_values('d_score', ascending=False)
cols_d = [
    'combination','target','n_features','d_score','rmse','r2','n_rows','n_subjects',
    'alpha_mode','l1_ratio_mode','alpha_mean','l1_ratio_mean',
]
cv_df_d_tuned.reindex(columns=cols_d)


,combination,target,n_features,d_score,rmse,r2,n_rows,n_subjects,alpha_mode,l1_ratio_mode,alpha_mean,l1_ratio_mean
16,background_structural_ext,FARS,21,1.584655,25.651969,0.021544,50,25,0.599484,0.9,1.018616,0.804000
9,background_structural,SARA,17,1.560755,7.601355,0.122662,47,25,0.215443,0.9,0.269808,0.580000
8,background_structural,FARS,17,1.526979,21.505606,0.312293,50,25,1.668101,0.9,1.420427,0.724000
17,background_structural_ext,SARA,21,1.419912,8.178118,-0.015528,47,25,0.599484,0.9,0.425966,0.788000
13,background_structural_diffusion,SARA,29,1.073344,6.811399,0.296717,46,25,1.668101,0.9,1.567250,0.884000
12,background_structural_diffusion,FARS,29,0.934822,22.186613,0.254739,49,25,4.641589,0.9,5.634458,0.868000
14,structural_ext_only,FARS,15,0.775654,24.364733,0.138166,52,26,1.668101,0.9,3.100544,0.853846
15,structural_ext_only,SARA,15,0.774785,7.642745,0.138810,49,26,0.599484,0.9,0.807609,0.884615
0,background_only,FARS,6,0.770741,23.572001,0.173786,50,25,12.915497,0.1,22.205062,0.180000
2,structural_only,FARS,11,0.735053,24.420392,0.134224,52,26,1.668101,0.1,1.494761,0.284615


In [25]:
# Save CV results
Path('results').mkdir(exist_ok=True)
cv_df.to_csv('results/cv_performance_full.csv', index=False)
cv_df[cv_df['target']=='FARS'].to_csv('results/cv_performance_fars.csv', index=False)
cv_df[cv_df['target']=='SARA'].to_csv('results/cv_performance_sara.csv', index=False)
print('Saved cv_performance_full.csv / fars / sara')

if 'cv_df_d_tuned' in globals():
    cv_df_d_tuned.to_csv('results/cv_performance_full_d_tuned.csv', index=False)
    cv_df_d_tuned[cv_df_d_tuned['target']=='FARS'].to_csv('results/cv_performance_fars_d_tuned.csv', index=False)
    cv_df_d_tuned[cv_df_d_tuned['target']=='SARA'].to_csv('results/cv_performance_sara_d_tuned.csv', index=False)
    print('Saved cv_performance_full_d_tuned.csv / fars / sara')


Saved cv_performance_full.csv / fars / sara
Saved cv_performance_full_d_tuned.csv / fars / sara


In [26]:
# Single-feature d benchmark (v2 - v1 paired)
single_d = []
for feat in structural + diffusion + background:
    if feat+'_v1' in melb_wide.columns and feat+'_v2' in melb_wide.columns:
        diffs = melb_wide[feat+'_v2'] - melb_wide[feat+'_v1']
        d = diffs.mean() / diffs.std(ddof=1) if diffs.std(ddof=1) != 0 else np.nan
        single_d.append({'feature': feat, 'd_score': d})

single_feature_d = pd.DataFrame(single_d).sort_values('d_score', ascending=False)
single_feature_d.to_csv('results/single_feature_d.csv', index=False)
single_feature_d.head()


,feature,d_score
13,FAICP,0.229760
22,RDICP,0.171096
11,FASCP,0.129891
16,MDICP,0.122145
19,ADICP,0.117491


In [27]:
# LDA direct d-maximization
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis

lda_rows = []
subjects = merged_long[subject_col].unique()

for combo in combinations:
    feats = combo['features']
    scores = []
    for subj in subjects:
        train = merged_long[merged_long[subject_col] != subj].dropna(subset=feats+['visit'])
        test = merged_long[merged_long[subject_col] == subj].dropna(subset=feats+['visit'])
        if len(test['visit'].unique()) < 2:
            continue
        lda = LinearDiscriminantAnalysis()
        lda.fit(train[feats], train['visit'])
        proj = lda.transform(test[feats]).ravel()
        v1 = proj[test['visit']==1][0]
        v2 = proj[test['visit']==2][0]
        scores.append(v2 - v1)
    d = np.mean(scores) / np.std(scores, ddof=1) if len(scores) > 1 else np.nan
    lda_rows.append({'combination': combo['name'], 'lda_d_score': d})

lda_df = pd.DataFrame(lda_rows)
lda_df.to_csv('results/lda_vs_elasticnet_d.csv', index=False)
lda_df.head()


,combination,lda_d_score
0,background_only,0.141751
1,structural_only,0.598246
2,diffusion_only,0.141262
3,all_neuroimaging,0.447946
4,background_structural,0.370914


In [28]:
# Feature selection frequency (ElasticNet)
# Use best combo (background_structural_diffusion) on FARS
best_key = ('background_structural_diffusion','FARS')
coef_mat = coef_dict.get(best_key)
if coef_mat is not None and coef_mat.size > 0:
    freq = (coef_mat != 0).mean(axis=0)
    feats = [f for f in combinations if f['name']=='background_structural_diffusion'][0]['features']
    feature_selection_frequency = pd.DataFrame({'feature': feats, 'selection_freq': freq})
    feature_selection_frequency.to_csv('results/feature_selection_frequency.csv', index=False)
    feature_selection_frequency.sort_values('selection_freq', ascending=False).head()


In [29]:
# Composite sensitivity from OOF predictions (no extra LOO)
key = ('background_structural_diffusion','FARS')
oof_df = oof_dict.get(key)
if oof_df is not None:
    paired = (oof_df.sort_values([subject_col,'visit'])
              .groupby(subject_col)['pred']
              .apply(list))
    paired = paired[paired.map(len)==2]
    diffs = paired.map(lambda x: x[1]-x[0])
    composite_sensitivity = {
        'n': len(diffs),
        'mean_change': diffs.mean(),
        'sd_change': diffs.std(ddof=1),
        'd_score': diffs.mean()/diffs.std(ddof=1) if diffs.std(ddof=1)!=0 else np.nan,
    }
    pd.DataFrame([composite_sensitivity]).to_csv('results/composite_sensitivity.csv', index=False)
    composite_sensitivity
